## Database: MongoDB

In [1]:
# ====================================================
# Installing PyMongo in Jupyter Notebook
# ====================================================

# !pip install pymongo
# ---------------------
# This command tells Python to download and install the 'pymongo' library.
# In Jupyter Notebook, the ! at the beginning allows you to run terminal commands directly from a notebook cell.

# PyMongo is required to:
# - Connect Python to MongoDB databases
# - Perform CRUD operations (Create, Read, Update, Delete) on MongoDB
# - Work with MongoDB documents (Python dictionaries)

# Run this cell ONCE before trying to use 'import pymongo'
#!pip install pymongo

In [2]:
# ====================================================
# Importing PyMongo and MongoDB Tools
# ====================================================

# 1. Import the main PyMongo library
# - PyMongo is the official Python library for working with MongoDB.
# - It allows Python to connect to MongoDB databases, send queries, and read/write documents.
import pymongo

# 2. Import MongoClient specifically
# - MongoClient is the class used to connect to a MongoDB server.
# - You create a MongoClient object to start talking to your database.
from pymongo import MongoClient

# 3. Import ObjectId from bson
# - MongoDB automatically assigns a unique ID (_id) to each document.
# - These IDs are of type ObjectId.
# - Importing ObjectId allows you to query documents by their _id field or create new ObjectIds.
from bson.objectid import ObjectId

In [3]:
# ====================================================
# Connect to MongoDB and Access a Collection
# ====================================================

# 1. Create a MongoClient
# - This object represents a connection to the MongoDB server.
# - "mongodb://localhost:27017/" tells Python to connect to a MongoDB server running
#   locally on your computer (localhost) using the default port 27017.
client = MongoClient("mongodb://localhost:27017/")

# 2. Access a specific database
# - MongoDB stores data in databases (like folders for collections).
# - Here, we are using a database called "CustomersDB".
# - If it doesn't exist, MongoDB will create it automatically when you first store data.
db = client["CustomersDB"]

# 3. Access a specific collection
# - Collections in MongoDB are similar to tables in SQL databases.
# - Here, we access the "Customers" collection inside "CustomersDB".
# - If it doesn't exist, MongoDB will create it automatically when you first insert a document.
customers_col = db["Customers"]

In [4]:
# ====================================================
# Function to Add a Customer to MongoDB
# ====================================================

def add_customer(customer):
    """
    This function adds ONE customer to the MongoDB database.

    Parameters:
    -----------
    customer : dict
        - A dictionary containing customer information
        - Example:
          {
              "name": "Ahmed",
              "age": 30,
              "gender": "Male",
              "region": "Cairo"
          }

    Returns:
    --------
    tuple (success, message)
        - success : True if insertion worked, False if an error occurred
        - message : Inserted ID on success, or error message on failure
    """

    try:
        # 1. Insert the customer dictionary into the MongoDB collection
        # - customers_col is the MongoDB collection object
        # - insert_one() returns a result object containing the inserted ID
        result = customers_col.insert_one({
            "Name": customer['name'],     # Customer name
            "Age": customer['age'],       # Customer age
            "Gender": customer['gender'], # Customer gender
            "Region": customer['region']  # Customer region
        })

        # 2. Return success and the inserted document ID
        return True, str(result.inserted_id)

    except Exception as e:
        # 3. If anything goes wrong, return False and the error message
        return False, str(e)

In [5]:
# ====================================================
# Example: Add a Customer to MongoDB
# ====================================================

# 1. Create a new customer object (dictionary)
# - Keys should match what the add_customer_mongo() function expects
new_customer = {
    "name": "Ahmed Fahmy",  # Customer's name
    "age": 35,              # Customer's age
    "gender": "Male",       # Customer's gender
    "region": "Cairo"       # Customer's region
}

# 2. Call the add_customer_mongo() function
# - Pass the customer dictionary
# - Function will insert the document into MongoDB
success, message = add_customer(new_customer)

# 3. Show the result
# - success: True if insert worked, False if there was an error
# - message: Inserted document ID or error message
print("Success:", success)
print("Message:", message)

Success: True
Message: 6960d5f96d3c09a97e8ddb64


In [6]:
# ====================================================
# Function to Get All Customers from MongoDB
# ====================================================

def get_customers():
    """
    This function retrieves ALL customers from the MongoDB collection.

    Returns:
    --------
    tuple (success, customers_or_error)
        - success : True if retrieval worked, False if there was an error
        - customers_or_error : List of customer dictionaries (with _id as string) on success,
                               or error message string on failure
    """

    try:
        # 1. Create an empty list to store customers
        customers = []

        # 2. Loop through each document in the MongoDB collection
        # - customers_col.find() retrieves all documents
        for doc in customers_col.find():
            # 3. Convert MongoDB ObjectId to string
            # - This makes it easier to display or use in JSON
            doc['_id'] = str(doc['_id'])

            # 4. Add the document (customer) to the list
            customers.append(doc)

        # 5. Return success and the list of customers
        return True, customers

    except Exception as e:
        # 6. If anything goes wrong, return False and the error message
        return False, str(e)

In [8]:
# ====================================================
# Example: Fetch All Customers from MongoDB
# ====================================================

# 1. Call the function to get all customers
# - get_customers_mongo() returns a tuple: (success flag, list of customers or error message)
success, customers = get_customers()

# 2. Check if fetching customers was successful
if success:
    # ✅ Print total number of customers
    print("Total customers:", len(customers))

    # Loop through each customer and print its details
    for c in customers:
        print(c)
else:
    # ❌ If fetching failed, print the error message
    print("Error fetching customers:", customers)

Total customers: 1
{'_id': '6960d5f96d3c09a97e8ddb64', 'Name': 'Ahmed Fahmy', 'Age': 35, 'Gender': 'Male', 'Region': 'Cairo'}


In [9]:
# ====================================================
# Function to Update a Customer in MongoDB
# ====================================================

def update_customer(customer_id, customer):
    """
    This function updates ONE customer in the MongoDB collection.

    Parameters:
    -----------
    customer_id : str
        - The _id of the MongoDB document (customer) to update

    customer : dict
        - A dictionary containing the updated customer information
        - Example:
          {
              "name": "Ahmed Ali",
              "age": 31,
              "gender": "Male",
              "region": "Giza"
          }

    Returns:
    --------
    tuple (success, message)
        - success : True if update worked, False if there was an error
        - message : Success message or error explanation
    """

    try:
        # 1. Update the MongoDB document using update_one()
        # - {"_id": ObjectId(customer_id)} → selects the document to update
        # - {"$set": {...}} → sets the new field values
        result = customers_col.update_one(
            {"_id": ObjectId(customer_id)},
            {"$set": {
                "Name": customer['name'],
                "Age": customer['age'],
                "Gender": customer['gender'],
                "Region": customer['region']
            }}
        )

        # 2. Check if any document matched the _id
        if result.matched_count == 0:
            return False, "Customer not found"

        # 3. Return success if update occurred
        return True, "Customer updated successfully"

    except Exception as e:
        # 4. Handle any errors (invalid ID, connection issues, etc.)
        return False, str(e)

In [10]:
# ====================================================
# Example: Update a Customer in MongoDB
# ====================================================

# 1. Prepare the updated customer data
# - This dictionary contains the new values we want to save
updated_customer = {
    "name": "Ahmed Fahmy Ali",  # Updated name
    "age": 36,                  # Updated age
    "gender": "Male",           # Gender remains the same
    "region": "Giza"            # Updated region
}

# 2. Specify the MongoDB _id of the customer to update
# - This must match the _id of an existing document in the collection
customer_id = "6960d5f96d3c09a97e8ddb64"

# 3. Call the update_customer_mongo() function
# - Pass the customer_id and updated_customer dictionary
success, message = update_customer(customer_id, updated_customer)

# 4. Show the result
# - success: True if update worked, False if something went wrong
# - message: Explanation of what happened
print("Success:", success)
print("Message:", message)

Success: True
Message: Customer updated successfully


In [12]:
# ====================================================
# Verify Customer Update in MongoDB
# ====================================================

# 1. Fetch all customers
success, customers = get_customers()

# 2. Check if fetching was successful
if success:
    # Loop through each customer to find the one we updated
    for c in customers:
        if c["_id"] == "6960d5f96d3c09a97e8ddb64":  # Replace with the actual _id of the updated customer
            print("Updated Customer:", c)
else:
    # Print the error if fetching failed
    print("Error fetching customers:", customers)

Updated Customer: {'_id': '6960d5f96d3c09a97e8ddb64', 'Name': 'Ahmed Fahmy Ali', 'Age': 36, 'Gender': 'Male', 'Region': 'Giza'}


In [13]:
# ====================================================
# Function to Delete a Customer from MongoDB
# ====================================================

def delete_customer(customer_id):
    """
    This function deletes ONE customer from the MongoDB collection.

    Parameters:
    -----------
    customer_id : str
        - The _id of the MongoDB document (customer) to delete

    Returns:
    --------
    tuple (success, message)
        - success : True if deletion worked, False if there was an error
        - message : Success message or error explanation
    """

    try:
        # 1. Delete the document with the given _id
        # - delete_one() deletes the first matching document
        result = customers_col.delete_one({"_id": ObjectId(customer_id)})

        # 2. Check if any document was actually deleted
        if result.deleted_count == 0:
            return False, "Customer not found"

        # 3. Return success if deletion occurred
        return True, "Customer deleted successfully"

    except Exception as e:
        # 4. Handle errors (invalid ObjectId, connection issues, etc.)
        return False, str(e)

In [14]:
# ====================================================
# Example: Delete a Customer in MongoDB
# ====================================================

# 1. Specify the MongoDB _id of the customer to delete
# - This must match an existing customer document
customer_id = "6960d5f96d3c09a97e8ddb64"  # Replace with the actual _id

# 2. Call the delete_customer_mongo() function
# - Pass the customer_id
# - Function will delete the customer if it exists
success, message = delete_customer(customer_id)

# 3. Show the result
print("Success:", success)
print("Message:", message)

Success: True
Message: Customer deleted successfully


In [16]:
# ====================================================
# Example: Verify Customer Deletion in MongoDB
# ====================================================

# 1. Fetch all remaining customers
# - get_customers_mongo() returns a tuple: (success flag, list of customers or error message)
success, customers = get_customers()

# 2. Check if fetching was successful
if success:
    # ✅ Print the total number of remaining customers
    print("Remaining customers:", len(customers))

    # Loop through each remaining customer and print details
    for c in customers:
        print(c)
else:
    # ❌ If fetching failed, print the error message
    print("Error fetching customers:", customers)

Remaining customers: 0
